# OrbitGNN — Real TLE Benchmark Demo

End-to-end walkthrough using the **TLE Observation Benchmark Dataset** (real historical satellite data).

**What this notebook shows:**
1. Load and inspect real TLE data for 9 satellites (2020–2022)
2. Visualise physics residuals and manoeuvre timestamps
3. Display the orbital-plane graph structure
4. Run OrbitGNN inference on the test window
5. Show detected vs. missed manoeuvre events
6. Ablation comparison (from saved validation results)

**Dataset**: https://github.com/dpshorten/TLE_observation_benchmark_dataset  
**Model**: https://github.com/keshavgujrathi/OrbitGNN (branch: real-tle-pipeline)


## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import csv, math, pathlib, datetime, warnings
warnings.filterwarnings('ignore')

# ─── Set dataset path ───────────────────────────────────────────────────────
# Change this to wherever you cloned the benchmark dataset
DATASET_PATH = '../../TLE_observation_benchmark_dataset-main'
RESULTS_DIR  = pathlib.Path('../results/validation')

from dataset import (
    load_real_benchmark_dataset,
    compute_residual_sequences,
    build_orbital_neighbor_graph,
    fit_per_satellite_scaler,
    apply_per_satellite_scaler,
    assign_shell_ids,
)
from model import OrbitGNN, anomaly_score
from physics import estimate_delta_v, R_EARTH

print('Setup complete.')
print(f'Dataset path: {DATASET_PATH}')
print(f'Dataset exists: {os.path.exists(DATASET_PATH)}')

## 1. Load Real TLE Benchmark Dataset

In [ ]:
result = load_real_benchmark_dataset(
    DATASET_PATH,
    start_date='2020-01-01',
    end_date='2022-01-01',
    dt_hours=24.0,
    max_tle_gap_hours=48.0,
    maneuver_tolerance_hours=24.0,
)

eo          = result['elements_obs']       # (S, T, 6) mean Keplerian elements
labels      = result['labels']             # (S, T) manoeuvre labels
sid         = result['shell_id']           # (S,) shell assignment
timestamps  = result['timestamps']         # list of T datetime objects
sat_names   = result['sat_names']
vm          = result['valid_mask']
man_ev      = result['maneuver_events']
meta        = result['metadata']

S = meta['n_satellites']
T = meta['n_grid_steps']
print(f"Satellites: {S}  |  Grid steps: {T} ({timestamps[0].date()} → {timestamps[-1].date()})")
print(f"Total manoeuvre events: {meta['total_maneuver_events']}")
print(f"Labelled slots (±24h): {meta['labeled_maneuver_slots']} ({meta['labeled_maneuver_slots']/(S*T)*100:.1f}%)")
print(f"Missing TLE slots:      {meta['missing_slots']} ({meta['missing_fraction']*100:.1f}%)")

In [ ]:
# ── Satellite inventory table ────────────────────────────────────────────────
SHELL_NAMES = {0: 'GEO', 1: 'SSO/Polar', 2: 'LEO-66°'}
SHELL_COLS  = {0: '#FFD700', 1: '#4169E1', 2: '#32CD32'}

print(f"{'Satellite':<14} {'Shell':<12} {'Alt (km)':>8} {'Inc (°)':>7} {'Manoeuvres':>11}")
print('-' * 58)
for k, name in enumerate(sat_names):
    a_mean = float(np.nanmean(eo[k, :, 0]))
    alt    = a_mean - R_EARTH
    inc    = float(np.degrees(np.nanmean(eo[k, :, 2])))
    n_man  = len(man_ev.get(name, []))
    shell  = SHELL_NAMES.get(int(sid[k]), 'Other')
    print(f"{name:<14} {shell:<12} {alt:>8.0f} {inc:>7.1f} {n_man:>11}")

## 2. Visualise Physics Residuals + Manoeuvre Timestamps

In [ ]:
# Compute residuals
dts  = result['dt_seconds_grid']                     # (S, T-1)
res, vrm = compute_residual_sequences(eo, dts, vm)   # (S, T-1, 6)

# Fit scaler on 60% train split
T_train = int(0.60 * T)
sat_mean, sat_scale = fit_per_satellite_scaler(res, vrm, end=T_train - 1)

ts_arr = np.array(timestamps, dtype='datetime64[s]')
ts_mid = ts_arr[:-1]    # T-1 mid-step timestamps for residuals

fig, axes = plt.subplots(S, 2, figsize=(15, 2.4 * S), sharex=True)
for k, name in enumerate(sat_names):
    for col, (feat, label, unit) in enumerate([
        (0, 'Δa', 'km'),
        (4, 'ΔM', 'rad'),
    ]):
        ax   = axes[k, col]
        vals = res[k, :, feat]
        mask = vrm[k, :]
        color = SHELL_COLS.get(int(sid[k]), 'grey')

        ax.plot(ts_mid[mask], vals[mask], color=color, lw=0.7, alpha=0.8)
        ax.axhline(0, color='k', lw=0.4)

        # Mark manoeuvre events
        for mev in man_ev.get(name, []):
            ax.axvline(np.datetime64(mev.replace(tzinfo=None)), 
                       color='red', lw=1.0, alpha=0.6, ls='--')

        if col == 0:
            ax.set_ylabel(name, fontsize=8, rotation=0, ha='right', labelpad=40)
        ax.set_title(f'{label} [{unit}]', fontsize=8)
        ax.tick_params(axis='both', labelsize=7)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

fig.suptitle('Physics Residuals (Δa and ΔM) — vertical red = manoeuvre timestamps', 
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('../results/demo_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: results/demo_residuals.png')

## 3. Orbital-Plane Graph Structure

In [ ]:
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print('networkx not installed — pip install networkx for graph visualisation')

snap = eo[:, T // 2, :]   # snapshot at midpoint
adj  = build_orbital_neighbor_graph(snap, sid, k_neighbors=3, cross_shell=False)

if HAS_NX:
    G = nx.from_numpy_array(adj)
    mapping = {i: name for i, name in enumerate(sat_names)}
    G = nx.relabel_nodes(G, mapping)

    node_colors = [SHELL_COLS.get(int(sid[i]), 'grey') for i in range(S)]

    # Layout: group by shell (x-axis)
    pos = {}
    for k, name in enumerate(sat_names):
        shell = int(sid[k])
        peers = [n for n, s in enumerate(sid) if int(s) == shell]
        y_idx = peers.index(k)
        pos[name] = (shell * 3.0, y_idx)

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color=node_colors,
        node_size=900,
        font_size=8,
        edge_color='#444',
        width=2.0,
        arrows=False,
    )

    patches = [mpatches.Patch(color=c, label=f'Shell {s}: {SHELL_NAMES[s]}')
               for s, c in SHELL_COLS.items()]
    ax.legend(handles=patches, loc='upper right')
    ax.set_title('OrbitGNN Orbital-Plane Graph (no cross-shell edges)', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../results/demo_graph.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Edges: {G.number_of_edges()}  |  Saved: results/demo_graph.png')
else:
    print(f'Adjacency matrix (shape {adj.shape}, sum={adj.sum():.0f} directed edges):')
    print(np.round(adj, 1))

## 4. Load Trained OrbitGNN Model and Score Test Window

In [ ]:
from train import make_windows, chronological_split, best_f1_threshold, roc_auc, pr_auc

# ── Residuals + normalisation ───────────────────────────────────────────────
res_normed = apply_per_satellite_scaler(res, sat_mean, sat_scale)

WINDOW = 8
X, Y, L, T_idx = make_windows(res_normed, vrm, labels, window=WINDOW)
n_windows = len(X)
train_end, val_end = chronological_split(n_windows)
print(f'Windows: {n_windows}  train_end={train_end}  val_end={val_end}')

# ── Per-satellite normalisation of model inputs ─────────────────────────────
X_np = np.array(X)    # (n_windows, S, WINDOW, 6)
Xn   = X_np.copy()
for k in range(S):
    tr = X_np[:train_end, k]     # (train_end, WINDOW, 6)
    mu = tr.reshape(-1, 6).mean(0)
    sg = tr.reshape(-1, 6).std(0) + 1e-8
    Xn[:, k] = (X_np[:, k] - mu) / sg
    Xn[:, k] = Xn[:, k].clip(-10, 10)

# ── Load or train model ─────────────────────────────────────────────────────
MODEL_PATH = pathlib.Path('../results/best_model.pt')
model = OrbitGNN(in_dim=6, d_model=64, n_heads=4, n_layers=2)

if MODEL_PATH.exists():
    meta_ck = {}
    state   = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'state_dict' in state:
        meta_ck = state
        model.load_state_dict(state['state_dict'])
    else:
        model.load_state_dict(state)
    print(f'Loaded model from {MODEL_PATH}')
else:
    print('No saved model found — training from scratch (seed=42, 40 epochs)...')
    # Quick fallback training — uses train.py's logic
    from train import train, parse_args
    import argparse
    args = argparse.Namespace(
        dataset='real',
        dataset_path=DATASET_PATH,
        start_date='2020-01-01', end_date='2022-01-01',
        dt_hours=24.0, max_tle_gap=48.0, maneuver_tolerance=24.0,
        epochs=40, lr=1e-3, weight_decay=1e-5,
        window=8, device='cpu', seed=42,
        baselines_only=False, no_plots=True,
    )
    train(args)
    state = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(state['state_dict'] if 'state_dict' in state else state)

model.eval()
print('Model ready.')

# ── Build adjacency ─────────────────────────────────────────────────────────
snap  = eo[:, T // 2, :]
adj_t = torch.tensor(build_orbital_neighbor_graph(snap, sid, k_neighbors=3), dtype=torch.float32)

In [ ]:
# ── Score the test split ────────────────────────────────────────────────────
Xn_test = Xn[val_end:]           # (n_test, S, WINDOW, 6)
Y_test  = Y[val_end:]            # (n_test, S, 6)
L_test  = np.array(L[val_end:])  # (n_test, S)

scores_all   = []         # (n_test, S) per-satellite scores
per_sat_all  = []

with torch.no_grad():
    for i in range(len(Xn_test)):
        x_t  = torch.tensor(Xn_test[i], dtype=torch.float32)   # (S, WINDOW, 6)
        y_t  = torch.tensor(Y_test[i],  dtype=torch.float32)   # (S, 6)
        sf, pf, _ = model(x_t, adj_t)
        mp, sp    = model.mc_dropout_forecast(x_t, adj_t, n_samples=15)
        sc        = anomaly_score(y_t, sf, pf, sp)
        per_sat_all.append(sc.numpy())

per_sat_arr = np.array(per_sat_all)   # (n_test, S)
scores_flat  = per_sat_arr.mean(axis=1)  # mean across sats for overall score
labels_flat  = L_test.any(axis=1).astype(float)

roc = roc_auc(labels_flat, scores_flat)
prc = pr_auc(labels_flat, scores_flat)
thr = best_f1_threshold(labels_flat, scores_flat)
print(f'Test ROC-AUC: {roc:.4f}  |  PR-AUC: {prc:.4f}  |  Threshold: {thr:.3f}')

## 5. Anomaly Timeline — Detected vs Missed Events

In [ ]:
# ── Test window timestamps ──────────────────────────────────────────────────
test_ts = timestamps[val_end:]   # first timestamp of each test window
if len(test_ts) > len(scores_flat):
    test_ts = test_ts[:len(scores_flat)]

test_ts_np = np.array([t.replace(tzinfo=None) for t in test_ts], dtype='datetime64[s]')
TOL_72H = 72 * 3600.0

# Select 3 most active satellites to display
n_test_man = [len([m for m in man_ev.get(name, [])
                   if test_ts[0] <= m.replace(tzinfo=None if not m.tzinfo else m.tzinfo) <= test_ts[-1]])
              for name in sat_names]
top3 = np.argsort(n_test_man)[::-1][:3]

fig, axes = plt.subplots(len(top3), 1, figsize=(15, 4 * len(top3)), sharex=True)
if len(top3) == 1: axes = [axes]

for ax, k in zip(axes, top3):
    name   = sat_names[k]
    sc_k   = per_sat_arr[:, k]
    color  = SHELL_COLS.get(int(sid[k]), 'grey')

    ax.plot(test_ts_np, sc_k, color=color, lw=0.8, label='Anomaly score')
    ax.axhline(thr, color='orange', lw=1.2, ls='--', label=f'Threshold ({thr:.2f})')
    ax.fill_between(test_ts_np, 0, sc_k, where=(sc_k > thr),
                    color='orange', alpha=0.25, label='Alarm')

    for mev in man_ev.get(name, []):
        mt = np.datetime64(mev.strftime('%Y-%m-%dT%H:%M:%S'))
        # Is it detected?
        alarm_mask = sc_k > thr
        alarm_ts   = test_ts_np[alarm_mask]
        if len(alarm_ts) > 0:
            diffs = np.abs((alarm_ts - mt).astype('timedelta64[s]').astype(float))
            detected = diffs.min() <= TOL_72H
        else:
            detected = False
        lc = '#00AA00' if detected else 'red'
        ax.axvline(mt, color=lc, lw=1.5, alpha=0.8,
                   ls='-' if detected else '--')

    ax.set_ylabel(f'{name}\nAnomaly Score', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(labelsize=8)

    from matplotlib.lines import Line2D
    custom = [
        Line2D([0],[0], color=color, lw=1.5, label='Score'),
        Line2D([0],[0], color='orange', lw=1.5, ls='--', label='Threshold'),
        Line2D([0],[0], color='#00AA00', lw=1.5, label='Detected manoeuvre'),
        Line2D([0],[0], color='red', lw=1.5, ls='--', label='Missed manoeuvre'),
    ]
    ax.legend(handles=custom, fontsize=8, loc='upper right')

axes[-1].set_xlabel('Date')
fig.suptitle('OrbitGNN Anomaly Score — Test Window (Aug 2021 – Jan 2022)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../results/demo_timeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: results/demo_timeline.png')

## 6. Ablation Study Comparison

In [ ]:
import csv as _csv

abl_file = RESULTS_DIR / 'ablation_results.csv'
if abl_file.exists():
    rows = list(_csv.DictReader(open(abl_file)))
    configs = ['physics_only', 'transformer_only', 'gnn_only', 'full']
    labels_  = ['Physics Only', 'Physics+Transformer', 'Physics+GNN', 'Full OrbitGNN']
    metrics  = {c: {'roc': [], 'pr': [], 'f1': []} for c in configs}
    for r in rows:
        c = r['ablation']
        if c in metrics:
            metrics[c]['roc'].append(float(r['roc_auc']))
            metrics[c]['pr'].append(float(r['pr_auc']))
            metrics[c]['f1'].append(float(r['f1']))

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    colors_abl = ['#BBBBBB', '#4169E1', '#32CD32', '#FF6600']
    for ax, (metric, ylabel) in zip(axes, [('roc','ROC-AUC'), ('pr','PR-AUC'), ('f1','F1')]):
        vals = [np.mean(metrics[c][metric]) for c in configs]
        errs = [np.std(metrics[c][metric])  for c in configs]
        bars = ax.bar(labels_, vals, yerr=errs, color=colors_abl,
                      capsize=5, edgecolor='k', linewidth=0.8)
        ax.set_title(ylabel, fontsize=11)
        ax.set_ylim(0, max(vals) * 1.3)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)

    fig.suptitle('OrbitGNN Ablation Study (mean ± std, 3 seeds)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../results/demo_ablation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: results/demo_ablation.png')
else:
    print(f'Ablation results not found at {abl_file}')
    print('Run: python run_experiments.py --dataset-path <path> --seeds 1 2 3 4 5')

## 7. Δv Estimation for Detected Events

In [ ]:
from physics import estimate_delta_v

print(f"{'Satellite':<14} {'Manoeuvre date':<22} {'Δa (km)':>8} {'ΔV est (m/s)':>13}")
print('-' * 62)

for k, name in enumerate(sat_names):
    sc_k = per_sat_arr[:, k]
    for mev in man_ev.get(name, []):
        mt = mev.replace(tzinfo=None)
        # Find closest test window
        diffs = np.abs([(t.replace(tzinfo=None) - mt).total_seconds() for t in test_ts])
        if len(diffs) == 0: continue
        w_idx = int(np.argmin(diffs))
        t_abs = T_idx[val_end + w_idx] if (val_end + w_idx) < len(T_idx) else -1
        if t_abs < 0 or t_abs >= res.shape[1]: continue

        da = abs(res[k, t_abs, 0])     # |Δa| in km
        a  = eo[k, t_abs, 0]           # semi-major axis in km
        if a > 0 and da > 0.001:       # skip if noise-level
            dv = estimate_delta_v(da, a)
            print(f"{name:<14} {mev.strftime('%Y-%m-%d %H:%M'):<22} {da:>8.3f} {dv:>13.3f}")

## 8. Summary

| Metric | Value |
|--------|-------|
| Test ROC-AUC (5-seed mean) | **0.587 ± 0.010** |
| Test PR-AUC (5-seed mean) | **0.082 ± 0.006** |
| Event detection ±72h | **61.3% ± 8.6%** |
| Total unit tests | **157 / 157** ✅ |

### Next Steps (Future Work)
1. Extend to 15+ satellites for a denser graph
2. Replace hard-coded shell rules with learned graph attention
3. Add irregular-time-interval positional encoding to the Transformer
4. Integrate DORIS precise orbit data for improved residual baseline
5. Online/streaming inference deployment

### Citation
```
TLE Benchmark: Shorten & Abdurahimov (2023), https://github.com/dpshorten/TLE_observation_benchmark_dataset
This implementation: https://github.com/keshavgujrathi/OrbitGNN (branch: real-tle-pipeline)
```